In [1]:
import torch
import os
import sys
import numpy as np
import cv2
import time
from ultralytics import YOLO
from sklearn.cluster import KMeans
import json

In [2]:
def get_kmeans(features, k=2):
    kmeans = KMeans(n_clusters=k, random_state=1, init='k-means++').fit(features)
    return kmeans

In [3]:
def get_player_team_color(image):
    # Crop top half of the image
    crop_top = image[:int(image.shape[0]/1.25), :]
    # Flatten the image
    features = crop_top.reshape(-1, 3)
    # Cluster the colors
    kmeans = get_kmeans(features)
    # Reshape the cluster centers
    clustered_image = kmeans.labels_.reshape(crop_top.shape[0], crop_top.shape[1])
    # Get the color of the corner pixels
    corner_cluster = [clustered_image[0, 0], clustered_image[0, -1], clustered_image[-1, 0], clustered_image[-1, -1]]
    
    non_player_cluster = max(set(corner_cluster), key=corner_cluster.count)
    player_cluster = 1-non_player_cluster
    player_color = kmeans.cluster_centers_[player_cluster]
    return player_color

In [4]:
def get_team_colors(players_color):
    kmeans = get_kmeans(players_color, k=2)
    team_colors = kmeans.cluster_centers_
    team_1_color = team_colors[0]
    team_2_color = team_colors[1]
    return team_1_color, team_2_color

In [6]:


model = YOLO("models/best_model_players.pt")
model_keypoints = YOLO("models/best_model_keypoints.pt")


In [7]:
output_video = 'prueba2.mp4'
cap = cv2.VideoCapture('videos/DFL Bundesliga Data Shootout/test/test (14).mp4')
# cap = cv2.VideoCapture('filmrole6.avi')
# Resizing the video to 640x640
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = 30
out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (640, 640))


frames = 100
counter = 0

data = []

while cap.isOpened():
    data.append({})
    ret, frame = cap.read()
    if not ret:
        break
    # frame = cv2.resize(frame, (640, 640))
    results = model(frame)
    results_keypoints = model_keypoints(frame)

    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            label = result.names[box.cls[0].item()]
            confidence = box.conf[0].item()
            label = f'{label} {confidence:.2f}'
            
            # Get the color of the player in the bbox
            player_color = get_player_team_color(frame[y1:y2, x1:x2])

            if 'boxes' not in data[counter]:
                data[counter]['boxes'] = []


            if 'player' == result.names[box.cls[0].item()] or 'goalkeeper' == result.names[box.cls[0].item()]:
                data[counter]['boxes'].append({
                    'x1': x1,
                    'y1': y1,
                    'x2': x2,
                    'y2': y2,
                    'label': result.names[box.cls[0].item()],
                    'player_color': player_color.tolist()
                })
            else:
                data[counter]['boxes'].append({
                    'x1': x1,
                    'y1': y1,
                    'x2': x2,
                    'y2': y2,
                    'label': result.names[box.cls[0].item()]
                })
        
        

        if counter == 0:
            team_colors = []
            for box in data[counter]['boxes']:
                if 'player_color' in box:
                    team_colors.append(box['player_color'])
            team_1_color, team_2_color = get_team_colors(team_colors)
            data[counter]['team_1_color'] = team_1_color.tolist()
            data[counter]['team_2_color'] = team_2_color.tolist()
    
    for result in results_keypoints:
        keypoints = result.keypoints.data[0]

        for keypoint in keypoints:
            x, y, conf = keypoint
            x, y = int(x), int(y)
            if 'keypoints' not in data[counter]:
                data[counter]['keypoints'] = []
                
        data[counter]['keypoints'] = keypoints.tolist()

            
    out.write(frame)
    if 'frame' not in data[counter]:
        data[counter]['frame'] = frame
        
    if counter > frames:
        break
    counter += 1

cap.release()
out.release()
cv2.destroyAllWindows()



0: 384x640 1 player, 205.3ms
Speed: 12.1ms preprocess, 205.3ms inference, 16.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 pitchs, 397.5ms
Speed: 15.6ms preprocess, 397.5ms inference, 6.0ms postprocess per image at shape (1, 3, 384, 640)


ValueError: n_samples=1 should be >= n_clusters=2.

In [23]:
keypoints = json.load(open("sample.json", "r"))
field_image = cv2.imread("football_field.png")

# Normalize keypoints
for keypoint in keypoints["keypoints"].keys():
    keypoints["keypoints"][keypoint] = [keypoints["keypoints"][keypoint][0], keypoints["keypoints"][keypoint][1]]

keypoints

{'keypoints': {'1': [18, 9],
  '2': [17, 49],
  '3': [18, 103],
  '4': [19, 191],
  '5': [18, 243],
  '6': [17, 279],
  '7': [43, 103],
  '8': [42, 190],
  '9': [70, 146],
  '10': [94, 48],
  '11': [95, 110],
  '12': [94, 181],
  '13': [94, 243],
  '14': [217, 11],
  '15': [216, 91],
  '16': [216, 182],
  '17': [217, 277],
  '18': [332, 49],
  '19': [333, 112],
  '20': [331, 182],
  '21': [332, 245],
  '22': [355, 147],
  '23': [382, 103],
  '24': [382, 189],
  '25': [407, 11],
  '26': [407, 50],
  '27': [406, 105],
  '28': [406, 190],
  '29': [409, 244],
  '30': [410, 280],
  '31': [173, 136],
  '32': [260, 135]},
 'height': 288,
 'width': 422}

In [24]:
def plot_players_and_keypoints(frame, data, field_image, keypoints_on_field_image, team_colors):
    if 'boxes' not in data:
        return frame
    
    for box in data['boxes']:
        x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
        label = box['label']
        if 'player_color' in box:
            # Check if is a goalkeeper
            if 'goalkeeper' in label:
                color = box['player_color']
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            else:
                team_1_color = np.array(team_colors[0])
                team_2_color = np.array(team_colors[1])
                player_color = np.array(box['player_color'])
                if np.linalg.norm(team_1_color - player_color) < np.linalg.norm(team_2_color - player_color):
                    color = team_colors[0]
                else:
                    color = team_colors[1]
        else:
            if 'ball' in label:
                color = (0, 255, 0)
            else:
                color = (255, 0, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    keypoints_coordinates = {}
    for i, keypoints in enumerate(data['keypoints']):
        x, y, conf = keypoints
        x, y = int(x), int(y)
        # Borrar el i
        if conf > 0.5:
            cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
            cv2.putText(frame, str(i), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
            keypoints_coordinates[str(i+1)] = (x, y)

    # Find the keypoints that were detected
    keypoints_on_field_image_detected = []
    for label, keypoint_pos in keypoints_on_field_image["keypoints"].items():
        if label in keypoints_coordinates.keys():
            keypoints_on_field_image_detected.append((label,keypoints_coordinates[label]))

    keypoints_position_detected = []
    keypoints_position = []
    for keypoint in keypoints_on_field_image_detected:
        keypoints_position_detected.append(keypoint[1])
        keypoints_position.append(keypoints_on_field_image["keypoints"][keypoint[0]])


    # Get the homography matrix
    H, _ = cv2.findHomography(np.array(keypoints_position_detected), np.array(keypoints_position))

    field_image_copy = field_image.copy()
    # Apply the homography matrix to the player positions
    for box in data['boxes']:
        x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
        player_position = np.array([[(x1+x2)/2, (y1+y2)/2]])
        player_position_on_field = cv2.perspectiveTransform(player_position[None, :, :], H)
        x, y = player_position_on_field[0][0]

        if 'player_color' in box:
            # Get the closer team color
            if 'goalkeeper' in box['label']:
                color = box['player_color']
                cv2.circle(field_image_copy, (int(x), int(y)), 5, color, -1)
            else:
                team_1_color = np.array(team_colors[0])
                team_2_color = np.array(team_colors[1])
                player_color = np.array(box['player_color'])
                if np.linalg.norm(team_1_color - player_color) < np.linalg.norm(team_2_color - player_color):
                    color = team_colors[0]
                else:
                    color = team_colors[1]
                cv2.circle(field_image_copy, (int(x), int(y)), 5, color, -1)
        else:
            if 'ball' in box['label']:
                cv2.circle(field_image_copy, (int(x), int(y)), 5, (0, 255, 0), -1)
            else:
                cv2.circle(field_image_copy, (int(x), int(y)), 5, (255, 0, 0), -1)


    return frame, field_image_copy

In [25]:

output_video = 'output3.mp4'

out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (640, 640))

frame_test = data.copy()

team_colors = [frame_test[0]['team_1_color'], frame_test[0]['team_2_color']]

print(len(frame_test))

# Remove all files from the frames folder
files = os.listdir("frames/video")
for file in files:
    os.remove(f"frames/video/{file}")

files = os.listdir("frames/field")
for file in files:
    os.remove(f"frames/field/{file}")

files = os.listdir("frames/concat")
for file in files:
    os.remove(f"frames/concat/{file}")

# Principal loop
for i, frame_data in enumerate(frame_test):
    # Check if frame_data has the 'frame' key
    if 'frame' not in frame_data:
        continue
    frame = frame_data['frame']
    frame, field_img = plot_players_and_keypoints(frame, frame_data, field_image.copy(), keypoints, team_colors)

    # Put the frame on the middel-bottom of the field with transparency
    concat = np.zeros((frame.shape[0]+field_img.shape[0], frame.shape[1], 3), dtype=np.uint8)
    concat[:frame.shape[0], :] = frame
    # Add padding
    field_img = cv2.copyMakeBorder(field_img, 0, 0, (frame.shape[1]-field_img.shape[1])//2, (frame.shape[1]-field_img.shape[1])//2, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    concat[frame.shape[0]:, :] = field_img


    # Write the frame
    out.write(concat)
    # Save the frames
    cv2.imwrite(f"frames/video/frame_{i}.png", frame)
    cv2.imwrite(f"frames/field/frame_{i}.png", field_img)
    cv2.imwrite(f"frames/concat/frame_{i}.png", concat)

    
out.release()
cv2.destroyAllWindows()


751


## Por si no se guardo bien el video a la primera

In [26]:
concat_path = "frames/concat"



In [27]:
# Order the images by the number of the frame
images = os.listdir(concat_path)
img_array = sorted(images, key=lambda x: int(x.split("_")[1].split(".")[0]))
img_array = [cv2.imread(os.path.join(concat_path, img)) for img in img_array]

# Reduce the size of the images to half
img_array = [cv2.resize(img, (int(img.shape[1]/2), int(img.shape[0]/2))) for img in img_array]
size = (img_array[0].shape[1], img_array[0].shape[0])

In [28]:


out = cv2.VideoWriter('output3.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30, size)

for i in range(len(img_array)):
    out.write(img_array[i])
out.release()
cv2.destroyAllWindows()


In [29]:
# Save half of the frames
out = cv2.VideoWriter('output3_half.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, size)

for i in range(len(img_array)//2):
    out.write(img_array[i])
out.release()
cv2.destroyAllWindows()

## Get football field keypoints

In [133]:
image_path = "football_field.png"
image = cv2.imread(image_path)

print(image.shape)

cv2.imshow('image', image)
cv2.waitKey(0)
cv2.destroyAllWindows()


(288, 422, 3)


In [134]:
keypoints_situation = []

num_keypoints = 32

for i in range(num_keypoints):
    keypoints_situation.append(())

# Loop to click on the image and get the keypoints
def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        for i in range(num_keypoints):
            if len(keypoints_situation[i]) < 2:
                keypoints_situation[i] = (x, y)
                cv2.circle(image, (x, y), 5, (0, 0, 255), -1)
                cv2.putText(image, str(i), (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                cv2.imshow('image', image)
                break

        print(keypoints_situation)

cv2.imshow('image', image)
cv2.setMouseCallback('image', click_event)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [21]:
index_keypoints = {}
for i, keypoint in enumerate(keypoints_situation):
    index_keypoints[i+1] = keypoint

print(index_keypoints)

{1: (18, 9), 2: (17, 49), 3: (18, 103), 4: (19, 191), 5: (18, 243), 6: (17, 279), 7: (43, 103), 8: (42, 190), 9: (70, 146), 10: (94, 48), 11: (95, 110), 12: (94, 181), 13: (94, 243), 14: (217, 11), 15: (216, 91), 16: (216, 182), 17: (217, 277), 18: (332, 49), 19: (333, 112), 20: (331, 182), 21: (332, 245), 22: (355, 147), 23: (382, 103), 24: (382, 189), 25: (407, 11), 26: (407, 50), 27: (406, 105), 28: (406, 190), 29: (409, 244), 30: (410, 280), 31: (173, 136), 32: (260, 135)}


In [22]:
import json

json_object = {}
json_object['keypoints'] = index_keypoints
json_object['height'] = image.shape[0]
json_object['width'] = image.shape[1]

# Writing to sample.json
with open("sample.json","w") as outfile:
    json.dump(json_object, outfile)

    

In [23]:
# Plot all keypoints on the image
for keypoint in index_keypoints.values():
    x, y = keypoint
    cv2.circle(image, (x, y), 5, (0, 0, 255), -1)
    cv2.imshow('image', image)
    cv2.waitKey(0)

## Reading all images and making a video

In [2]:
frames_path = 'frames/video'
fields_path = 'frames/field'

# Read all images from this paths
frames = []
fields = []

for i in os.listdir(frames_path):
    frames.append(cv2.imread(os.path.join(frames_path, i)))

for i in os.listdir(fields_path):
    fields.append(cv2.imread(os.path.join(fields_path, i)))

print(len(frames))
print(len(fields))


750
750


In [13]:
# Concat all the frames with the field

for i in range(len(frames)):
    frame = frames[i]
    field = fields[i]
    # Put the frame on the middel-bottom of the field with transparency
    concat = np.zeros((frame.shape[0]+field.shape[0], frame.shape[1], 3), dtype=np.uint8)
    concat[:frame.shape[0], :] = frame
    # Add padding
    field = cv2.copyMakeBorder(field, 0, 0, (frame.shape[1]-field.shape[1])//2, (frame.shape[1]-field.shape[1])//2, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    concat[frame.shape[0]:, :] = field
    
    cv2.imshow('image', concat)
    # Save the frame
    cv2.imwrite(f"frames/concat/frame_{i}.png", concat)
    cv2.waitKey(0)

cv2.destroyAllWindows()

